# Point density EDA — how many points a trip has, and why

Answers three questions for any city in the gold format:

1. **How far apart are the points?** distance, time and speed between neighbouring
   fixes, and how many points a trip carries per km and per minute.
2. **What decides where a point is?** Is it the road geometry re-emitted, a fixed
   time cadence, a fixed distance cadence, or nothing in particular.
3. **What does distance-resampling change?** point counts with and without the
   filter, and — the reason for all of this — how strongly `n_points` still
   predicts `Total_time`.

### Why it matters

If fixes arrive on a clock, then `n_points ≈ Total_time / cadence`, and a model
reads the answer off the sequence length instead of learning anything about the
road. Resampling every N **metres** breaks that: the point count becomes a
function of distance, which the model is supposed to know anyway.

### The filter implemented here

For every pair of neighbouring fixes:

* **same `OSMid`, edge present in the network, both points project close to the
  line** → cut the piece of the edge polyline *between the two projections*
  (not the straight line between the raw fixes);
* **otherwise** — different `OSMid` (an intersection), edge missing, or a
  projection far from the geometry (a likely map-matching error) — → honest
  straight line between the two raw fixes.

Those pieces are welded into one dense polyline per trip, with timestamps
interpolated proportionally to the distance covered inside each original sampling
window. The dense polyline is then walked from the start and a point is kept every
`RESAMPLE_STEP_M` metres of arc length: anything nearer than the step is dropped,
and the point that crosses it is projected back onto the exact mark. Both ends of
the trip are always kept, so `Total_time` is unchanged.

**Nothing here is run on real data by default** — set `CITY`, point `DATA_DIR` at
the files, and run. The last cell is a self-contained synthetic check that needs
no data at all.

## Configuration

In [ ]:
# ---------------------------------------------------------------- pick a city
CITY     = "omsk"
DATA_DIR = "."

# Files follow the gold naming by default: matched_trips_<city>.csv,
# road_network_unique_osmids_<city>.geojson, edge_list_directed_<city>.csv.
# Add an entry only when a city breaks that convention.
DATASETS = {
    "omsk":   {"trips": "matched_trips_omsk_mini.csv",
               "geojson": "road_network_unique_osmids_omsk.geojson"},
    "harbin": {"trips": "matched_trips_harbin_mini.csv",
               "geojson": "Harbin_road_network_unique_osmids.geojson"},
    # "ann_arbor", "rome", "san_francisco", "beijing_geolife", "athens",
    # "luxembourg", "nyc_citibike" all follow the convention — nothing to add.
}

RESAMPLE_STEP_M  = 500.0   # keep one point every this many metres of arc length
MAX_PROJ_DIST_M  = 50.0    # further than this from the matched edge = bad match
VERTEX_EPS_M     = 1.0     # "the fix sits on a vertex of the edge polyline"
DT_TOL_S         = 1.0     # tolerance for "the same time gap"
STEP_TOL_M       = 5.0     # tolerance for "the same distance gap"

MAX_TRIPS        = 2000    # None = all; the resampler is pure Python, so cap it
RANDOM_SEED      = 0
SAVE_RESAMPLED   = False   # write matched_trips_<city>_every<step>m.csv at the end

In [ ]:
# pip install pandas numpy shapely matplotlib tqdm   (pyproj optional but better)
import ast, json, math, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from shapely.geometry import LineString, Point
from shapely.ops import substring
from tqdm.auto import tqdm

# validated two-series palette (blue = before, orange = after)
C_BEFORE, C_AFTER, C_INK, C_GRID = "#2a78d6", "#eb6834", "#52514e", "#e6e5e1"
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.color": C_GRID,
                     "axes.edgecolor": C_GRID, "axes.labelcolor": C_INK,
                     "xtick.color": C_INK, "ytick.color": C_INK,
                     "axes.spines.top": False, "axes.spines.right": False})

## Loading

In [ ]:
# ---------------------------------------------------------------- loading
def load_trips(path, max_trips=None, seed=0):
    """Read a gold-format matched_trips_*.csv and parse the three list columns."""
    df = pd.read_csv(path)
    if max_trips and len(df) > max_trips:
        df = df.sample(max_trips, random_state=seed).sort_index()

    coords, osmids, stamps = [], [], []
    for c, o, t in zip(df["Coordinates"], df["OSMids"], df["Timestamps"]):
        coords.append(ast.literal_eval(c))
        osmids.append([str(x) for x in ast.literal_eval(o)])
        stamps.append([int(x) for x in ast.literal_eval(t)])
    out = pd.DataFrame({"Id": df["Id"].values, "coords": coords,
                        "osmids": osmids, "times": stamps,
                        "Total_time": df["Total_time"].values})
    bad = out.apply(lambda r: not (len(r.coords) == len(r.osmids) == len(r.times)), axis=1)
    if bad.any():
        print(f"warning: dropping {int(bad.sum())} trips with unequal list lengths")
        out = out[~bad]
    has_geom = out.coords.map(lambda c: all(p[0] is not None for p in c))
    if not has_geom.all():
        print(f"warning: {int((~has_geom).sum())} trips carry no coordinates "
              "(Quebec-style placeholder) — they cannot be resampled")
        out = out[has_geom]
    return out.reset_index(drop=True)


def load_edges(geojson_path):
    """unique segment id -> shapely LineString in lon/lat."""
    if not geojson_path or not os.path.exists(geojson_path):
        print("no road-network geojson — every pair will fall back to a straight line")
        return {}
    with open(geojson_path) as f:
        gj = json.load(f)
    out = {}
    for feat in gj["features"]:
        p = feat["properties"]
        key = str(p.get("unique_osmid", p.get("osmid")))
        geom = feat["geometry"]
        if geom["type"] != "LineString" or len(geom["coordinates"]) < 2:
            continue
        out[key] = LineString([(c[0], c[1]) for c in geom["coordinates"]])
    print(f"{len(out)} road segments with geometry")
    return out


# ---------------------------------------------------------------- projection
def make_projector(lon0, lat0):
    """(lon, lat) <-> local metres.  UTM via pyproj, equirectangular if absent."""
    try:
        from pyproj import CRS, Transformer
        zone = int((lon0 + 180) // 6) + 1
        crs = CRS.from_dict({"proj": "utm", "zone": zone, "south": lat0 < 0,
                             "ellps": "WGS84", "datum": "WGS84", "units": "m"})
        fwd = Transformer.from_crs("EPSG:4326", crs, always_xy=True).transform
        inv = Transformer.from_crs(crs, "EPSG:4326", always_xy=True).transform
        print(f"projection: UTM zone {zone}")
        return fwd, inv
    except ImportError:
        R, k = 6371000.0, math.cos(math.radians(lat0))
        print("projection: local equirectangular (pyproj not installed)")

        def fwd(lon, lat):
            lon, lat = np.asarray(lon, float), np.asarray(lat, float)
            return R * k * np.radians(lon), R * np.radians(lat)

        def inv(x, y):
            x, y = np.asarray(x, float), np.asarray(y, float)
            return np.degrees(x / (R * k)), np.degrees(y / R)
        return fwd, inv


def project_line(line, fwd):
    xs, ys = fwd(*np.asarray(line.coords).T[:2])
    return LineString(np.column_stack([xs, ys]))


# ---------------------------------------------------------------- geometry
def seg_lengths(xy):
    d = np.diff(np.asarray(xy, float), axis=0)
    return np.hypot(d[:, 0], d[:, 1])


# ---------------------------------------------------------------- densify
def densify_trip(xy, osmids, times, edges_m, max_proj_m=50.0, snap_eps=0.5):
    """Rebuild a trip as a dense polyline that follows the matched road geometry.

    For every pair of neighbouring fixes:
      * same OSMid, edge known, both projections close to the line
            -> cut the piece of the edge polyline between the two projections;
      * otherwise (intersection, unknown edge, or a projection far from the
        geometry, i.e. a likely map-matching error)
            -> honest straight line between the two original fixes.

    Timestamps inside a window are interpolated proportionally to distance
    travelled, so the dense polyline keeps the original per-window timing.
    """
    xy = np.asarray(xy, float)
    out_xy, out_t, out_oid = [xy[0]], [float(times[0])], [osmids[0]]
    kinds = {"edge": 0, "straight": 0}

    for i in range(len(xy) - 1):
        p0, p1 = xy[i], xy[i + 1]
        t0, t1 = float(times[i]), float(times[i + 1])
        piece, oid = None, osmids[i]

        if osmids[i] == osmids[i + 1] and osmids[i] in edges_m:
            line = edges_m[osmids[i]]
            a, b = Point(p0), Point(p1)
            if max(line.distance(a), line.distance(b)) <= max_proj_m:
                s0, s1 = line.project(a), line.project(b)
                if abs(s1 - s0) > 1e-6:
                    sub = substring(line, s0, s1)
                    if sub.geom_type == "LineString" and len(sub.coords) >= 2:
                        piece = np.asarray(sub.coords, float)[:, :2]

        if piece is None:
            piece = np.array([p0, p1])
            kinds["straight"] += 1
        else:
            kinds["edge"] += 1

        # the projected piece may not start exactly where the previous one ended;
        # keep the polyline continuous instead of silently teleporting
        tail = out_xy[-1]
        new = piece if np.hypot(*(piece[0] - tail)) > snap_eps else piece[1:]
        if len(new) == 0:
            continue

        step = seg_lengths(np.vstack([tail, new]))
        cum = np.cumsum(step)
        frac = cum / cum[-1] if cum[-1] > 0 else np.linspace(0, 1, len(cum))
        out_xy.extend(new)
        out_t.extend(t0 + frac * (t1 - t0))
        out_oid.extend([oid] * len(new))

    out_xy = np.asarray(out_xy, float)
    out_t = np.asarray(out_t, float)
    # time must not run backwards after interpolation
    out_t = np.maximum.accumulate(out_t)
    return out_xy, out_t, out_oid, kinds


# ---------------------------------------------------------------- resample
def resample_by_distance(xy, t, oid, step_m=500.0):
    """Emit a point every `step_m` along the polyline; always keep both ends.

    This is the "walk forward, drop everything nearer than `step_m`, project the
    next point onto the exact `step_m` mark" rule, done by arc length.

    Returns (points, times, osmids, marks); `marks` is the arc length at each
    kept point, so `diff(marks) == step_m` exactly.
    """
    xy = np.asarray(xy, float)
    seg = seg_lengths(xy)
    cum = np.concatenate([[0.0], np.cumsum(seg)])
    total = cum[-1]
    if total <= 0:
        return (xy[[0, -1]], np.asarray([t[0], t[-1]], float),
                [oid[0], oid[-1]], np.zeros(2))

    marks = np.arange(0.0, total, step_m)
    if total - marks[-1] > 1e-6:
        marks = np.append(marks, total)

    idx = np.clip(np.searchsorted(cum, marks, side="right") - 1, 0, len(seg) - 1)
    denom = np.where(seg[idx] > 0, seg[idx], 1.0)
    f = (marks - cum[idx]) / denom

    pts = xy[idx] + (xy[idx + 1] - xy[idx]) * f[:, None]
    ts = np.asarray(t)[idx] + (np.asarray(t)[idx + 1] - np.asarray(t)[idx]) * f
    oids = [oid[min(i + 1, len(oid) - 1)] for i in idx]
    # `marks` is the arc length at each kept point: consecutive values differ by
    # exactly step_m.  The straight-line gap is shorter wherever the road bends.
    return pts, ts, oids, marks

In [ ]:
cfg = DATASETS.get(CITY, {})
trips_path = os.path.join(DATA_DIR, cfg.get("trips", f"matched_trips_{CITY}.csv"))
geojson_path = os.path.join(
    DATA_DIR, cfg.get("geojson", f"road_network_unique_osmids_{CITY}.geojson"))

trips = load_trips(trips_path, max_trips=MAX_TRIPS, seed=RANDOM_SEED)
print(f"{len(trips)} trips from {trips_path}")

edges = load_edges(geojson_path)

# one local metric projection for the whole city, taken from the first trip
first = np.asarray(trips.coords.iloc[0], float)
fwd, inv = make_projector(float(first[:, 0].mean()), float(first[:, 1].mean()))
edges_m = {k: project_line(v, fwd) for k, v in edges.items()}

## 1. How far apart are the points?

Two tables: one row per *pair* of neighbouring fixes, and one row per *trip*.

In [ ]:
# ---------------------------------------------------------------- statistics
def pair_frame(trips, fwd):
    """One row per pair of neighbouring fixes: dt, step in metres, speed."""
    dt, step, tid, oid_same = [], [], [], []
    for r in trips.itertuples():
        X, Y = fwd(*np.asarray(r.coords, float).T)
        d = seg_lengths(np.column_stack([X, Y]))
        t = np.diff(np.asarray(r.times, float))
        dt.append(t); step.append(d)
        tid.append(np.full(len(t), r.Index))
        oid_same.append(np.asarray(r.osmids[:-1]) == np.asarray(r.osmids[1:]))
    out = pd.DataFrame({"trip": np.concatenate(tid), "dt_s": np.concatenate(dt),
                        "step_m": np.concatenate(step),
                        "same_osmid": np.concatenate(oid_same)})
    out["speed_ms"] = out.step_m / out.dt_s.replace(0, np.nan)
    return out


def trip_frame(trips, pairs):
    g = pairs.groupby("trip")
    out = pd.DataFrame({
        "n_points": trips.coords.map(len).values,
        "duration_s": trips.Total_time.values,
        "path_m": g.step_m.sum().reindex(range(len(trips)), fill_value=0).values,
        "n_segments": trips.osmids.map(lambda o: len(set(o))).values,
    })
    out["points_per_km"] = out.n_points / (out.path_m / 1000).replace(0, np.nan)
    out["points_per_min"] = out.n_points / (out.duration_s / 60).replace(0, np.nan)
    out["mean_speed_ms"] = out.path_m / out.duration_s.replace(0, np.nan)
    return out


def _regularity(v, tol, name, unit, quiet=False):
    """How close a sample is to 'one fixed value everywhere'.

    `share`   - fraction within +-tol of the modal value.
    `rel_iqr` - (q75-q25)/median, a dispersion measure that a heavy tail of
                long gaps cannot fake.  Small rel_iqr => a fixed cadence.
    """
    v = np.asarray(v, float)
    v = v[np.isfinite(v)]
    if len(v) == 0:
        return {"mode": np.nan, "share": 0.0, "rel_iqr": np.nan, "median": np.nan}
    mode = float(pd.Series(np.round(v / tol) * tol).mode().iloc[0])
    share = float(np.mean(np.abs(v - mode) <= tol))
    q25, med, q75 = np.percentile(v, [25, 50, 75])
    rel_iqr = float((q75 - q25) / med) if med else np.inf
    if not quiet:
        print(f"  {name:<24} mode {mode:>7.1f} {unit}  median {med:>7.1f}  "
              f"within +-{tol:g}: {share:6.1%}  rel.IQR {rel_iqr:5.2f}")
    return {"mode": mode, "share": share, "rel_iqr": rel_iqr, "median": float(med)}


def _is_regular(stat, share_min=0.5, iqr_max=0.35):
    return stat["share"] >= share_min or stat["rel_iqr"] <= iqr_max


def vertex_share(trips, edges_m, fwd, eps_m=1.0, max_trips=300):
    """Share of fixes sitting on a *vertex* of their own matched edge.

    High share => the 'GPS points' are really the road polyline, re-emitted.
    """
    if not edges_m:
        print("  road geometry unavailable — vertex test skipped")
        return np.nan
    hit = tot = 0
    for r in trips.head(max_trips).itertuples():
        X, Y = fwd(*np.asarray(r.coords, float).T)
        for x, y, o in zip(X, Y, r.osmids):
            line = edges_m.get(o)
            if line is None:
                continue
            v = np.asarray(line.coords, float)[:, :2]
            tot += 1
            if np.min(np.hypot(v[:, 0] - x, v[:, 1] - y)) <= eps_m:
                hit += 1
    return hit / tot if tot else np.nan


def diagnose(pairs, trips_stats, trips, edges_m, fwd,
             dt_tol=1.0, step_tol=5.0, vertex_eps_m=1.0):
    print("regularity of the spacing between neighbouring fixes")
    t = _regularity(pairs.dt_s, dt_tol, "time between fixes", "s")
    d = _regularity(pairs.step_m, step_tol, "distance between fixes", "m")

    dup = float((pairs.step_m < 1e-6).mean())
    zero_dt = float((pairs.dt_s == 0).mean())

    # duplicated vertices drown the real cadence; measure it again without them
    moving = pairs[(pairs.step_m > 1e-6) & (pairs.dt_s > 0)]
    t_mv, d_mv = t, d
    if dup > 0.05 or zero_dt > 0.05:
        print("\n  ... the same, counting only pairs that actually move in space and time")
        t_mv = _regularity(moving.dt_s, dt_tol, "time between fixes", "s")
        d_mv = _regularity(moving.step_m, step_tol, "distance between fixes", "m")
    vs = vertex_share(trips, edges_m, fwd, eps_m=vertex_eps_m)

    print(f"\n  duplicate consecutive coordinates : {dup:.1%}")
    print(f"  zero time between fixes           : {zero_dt:.1%}")
    if np.isfinite(vs):
        print(f"  fixes sitting on an edge vertex   : {vs:.1%} "
              f"(within {vertex_eps_m:g} m)")

    ok = trips_stats.dropna(subset=["path_m", "duration_s"])
    r_time = ok.n_points.corr(ok.duration_s)
    r_dist = ok.n_points.corr(ok.path_m)
    print(f"\n  corr(n_points, duration) = {r_time:+.3f}")
    print(f"  corr(n_points, path length) = {r_dist:+.3f}")

    print("\nverdict")
    geometry = np.isfinite(vs) and vs > 0.5
    if geometry:
        print("  * the coordinates are ROAD GEOMETRY, not raw GPS: most fixes land")
        print("    exactly on a vertex of their own matched edge. Point count is")
        print("    then a property of how finely OSM drew the street.")
    if _is_regular(t_mv):
        print(f"  * TIME-based sampling, about every {t_mv['median']:.0f} s "
              f"(rel.IQR {t_mv['rel_iqr']:.2f}).")
    elif _is_regular(d_mv):
        print(f"  * DISTANCE-based sampling, about every {d_mv['median']:.0f} m "
              f"(rel.IQR {d_mv['rel_iqr']:.2f}).")
    elif not geometry:
        print("  * IRREGULAR: neither the time nor the distance between fixes")
        print("    concentrates on one value.")
    if dup > 0.05:
        print(f"  * {dup:.0%} of neighbouring fixes share the same coordinate — "
              "repeated\n    vertices at edge boundaries, not real standstills.")
    if abs(r_time) - abs(r_dist) > 0.15:
        print("  * n_points tracks DURATION more than distance: a model can read")
        print("    the answer off the sequence length. This is the leak to remove.")
    elif abs(r_dist) - abs(r_time) > 0.15:
        print("  * n_points tracks DISTANCE more than duration — no direct leak.")
    return {"dt": t, "step": d, "dt_moving": t_mv, "step_moving": d_mv,
            "dup": dup, "zero_dt": zero_dt, "vertex_share": vs,
            "corr_time": r_time, "corr_dist": r_dist}

In [ ]:
pairs = pair_frame(trips, fwd)
print(f"{len(pairs):,} pairs of neighbouring fixes\n")
pairs[["dt_s", "step_m", "speed_ms"]].describe(
    percentiles=[.05, .25, .5, .75, .95]).round(2)

In [ ]:
trip_stats = trip_frame(trips, pairs)
trip_stats[["n_points", "duration_s", "path_m", "n_segments",
            "points_per_km", "points_per_min", "mean_speed_ms"]].describe(
    percentiles=[.05, .25, .5, .75, .95]).round(2)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 3.4))

d = pairs.dt_s[pairs.dt_s.between(0, pairs.dt_s.quantile(0.99))]
ax[0].hist(d, bins=60, color=C_BEFORE, edgecolor="none")
ax[0].set_title("time between neighbouring fixes", color=C_INK, loc="left")
ax[0].set_xlabel("seconds")

s = pairs.step_m[pairs.step_m.between(0, pairs.step_m.quantile(0.99))]
ax[1].hist(s, bins=60, color=C_BEFORE, edgecolor="none")
ax[1].set_title("distance between neighbouring fixes", color=C_INK, loc="left")
ax[1].set_xlabel("metres")

for a in ax:
    a.set_ylabel("pairs")
    a.set_axisbelow(True)
fig.suptitle(f"{CITY} — spacing of the raw fixes (99th percentile clipped)",
             color=C_INK, x=0.02, ha="left")
fig.tight_layout()

## 2. What decides where a point is?

Three candidate explanations, and a test for each:

* **road geometry** — the "fixes" are the vertices of the matched edge, re-emitted
  with interpolated times. Tested by asking what share of fixes sit on a vertex of
  their *own* matched edge.
* **a clock** — one fix every N seconds. Tested on the spread of `dt`.
* **an odometer** — one fix every N metres. Tested on the spread of `step_m`.

`rel.IQR` = (q75−q25)/median. A heavy tail of long gaps inflates the mean and the
CV but leaves `rel.IQR` small, so it is the honest measure of "one fixed cadence".
Duplicated vertices and zero-length gaps are measured separately and the cadence is
then re-measured without them, because they otherwise drown the real signal.

In [ ]:
diag = diagnose(pairs, trip_stats, trips, edges_m, fwd,
                dt_tol=DT_TOL_S, step_tol=STEP_TOL_M, vertex_eps_m=VERTEX_EPS_M)

In [ ]:
# eyeball a few trips: the raw cadence, fix by fix
for r in trips.head(3).itertuples():
    dt = np.diff(np.asarray(r.times, float))[:14]
    X, Y = fwd(*np.asarray(r.coords, float).T)
    dm = seg_lengths(np.column_stack([X, Y]))[:14]
    print(f"trip {str(r.Id)[:18]:<18} n={len(r.coords):>4}  "
          f"segments={len(set(r.osmids)):>4}")
    print("   dt (s) ", np.round(dt, 1).tolist())
    print("   step(m)", np.round(dm, 1).tolist())

## 3. The filter — follow the road, then keep one point every N metres

In [ ]:
# ---------------------------------------------------------------- pipeline
def resample_dataset(trips, edges_m, fwd, inv, step_m=500.0, max_proj_m=50.0):
    """Densify along the matched geometry, then keep one point every `step_m`."""
    rows, kinds = [], {"edge": 0, "straight": 0}
    dense_m, raw_m = [], []

    for r in tqdm(list(trips.itertuples()), desc="resampling"):
        X, Y = fwd(*np.asarray(r.coords, float).T)
        xy = np.column_stack([X, Y])
        raw_m.append(seg_lengths(xy).sum())

        dxy, dt_, doid, k = densify_trip(xy, r.osmids, r.times, edges_m, max_proj_m)
        kinds["edge"] += k["edge"]
        kinds["straight"] += k["straight"]
        dense_m.append(seg_lengths(dxy).sum())

        rxy, rt, roid, marks = resample_by_distance(dxy, dt_, doid, step_m)
        lon, lat = inv(rxy[:, 0], rxy[:, 1])
        stamps = np.maximum.accumulate(np.round(rt).astype("int64")).tolist()
        rows.append({
            "Id": r.Id,
            "Coordinates": str([(round(float(a), 6), round(float(b), 6))
                                for a, b in zip(np.atleast_1d(lon), np.atleast_1d(lat))]),
            "OSMids": str([str(o) for o in roid]),
            "Timestamps": str(stamps),
            "Total_time": stamps[-1] - stamps[0],
        })

    out = pd.DataFrame(rows, columns=["Id", "Coordinates", "OSMids",
                                      "Timestamps", "Total_time"])
    total = kinds["edge"] + kinds["straight"]
    print(f"\npairs followed along the edge polyline : {kinds['edge']:>9,} "
          f"({kinds['edge']/total:.1%})" if total else "no pairs")
    print(f"pairs fallen back to a straight line   : {kinds['straight']:>9,} "
          f"({kinds['straight']/total:.1%})" if total else "")
    print(f"path length: raw {np.sum(raw_m)/1000:,.0f} km -> "
          f"densified {np.sum(dense_m)/1000:,.0f} km "
          f"({np.sum(dense_m)/max(np.sum(raw_m), 1e-9) - 1:+.1%})")
    return out, kinds


def compare(before_trips, before_stats, after_df, step_m):
    """a) all points  b) points after filtering — and what it does to the leak."""
    n_before = before_stats.n_points
    n_after = pd.Series([len(ast.literal_eval(c)) for c in after_df.Coordinates])
    dur = before_stats.duration_s.reset_index(drop=True)
    path = before_stats.path_m.reset_index(drop=True)

    tbl = pd.DataFrame({
        "trips": [len(n_before), len(n_after)],
        "points_total": [int(n_before.sum()), int(n_after.sum())],
        "points_mean": [n_before.mean(), n_after.mean()],
        "points_median": [n_before.median(), n_after.median()],
        "points_min": [n_before.min(), n_after.min()],
        "points_max": [n_before.max(), n_after.max()],
        "corr_with_duration": [n_before.corr(dur), n_after.corr(dur)],
        "corr_with_path": [n_before.corr(path), n_after.corr(path)],
    }, index=["all points", f"every {step_m:.0f} m"]).T
    tbl["change"] = tbl.iloc[:, 1] / tbl.iloc[:, 0].replace(0, np.nan)
    return tbl.round(3), n_before, n_after

In [ ]:
resampled, kinds = resample_dataset(trips, edges_m, fwd, inv,
                                    step_m=RESAMPLE_STEP_M,
                                    max_proj_m=MAX_PROJ_DIST_M)
resampled.head(3)

## 4. Counts, before and after

`corr_with_duration` is the number this whole exercise exists to move. Before the
filter it says how much of `Total_time` a model can read straight off the sequence
length; after, `n_points` is a function of distance by construction, so
`corr_with_path` goes to 1 and the duration leak drops to whatever correlation
distance and duration genuinely have.

In [ ]:
table, n_before, n_after = compare(trips, trip_stats, resampled, RESAMPLE_STEP_M)
table

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(14, 3.6))

hi = int(np.percentile(np.r_[n_before, n_after], 99))
bins = np.linspace(0, max(hi, 2), 50)
ax[0].hist(n_before, bins=bins, color=C_BEFORE, alpha=0.85,
           edgecolor="none", label="all points")
ax[0].hist(n_after, bins=bins, color=C_AFTER, alpha=0.85,
           edgecolor="none", label=f"every {RESAMPLE_STEP_M:.0f} m")
ax[0].set_xlabel("points per trip")
ax[0].set_ylabel("trips")
ax[0].set_title("points per trip", color=C_INK, loc="left")
ax[0].legend(frameon=False, labelcolor=C_INK)

for a, x, xlabel in ((ax[1], trip_stats.duration_s, "trip duration (s)"),
                     (ax[2], trip_stats.path_m / 1000, "path length (km)")):
    a.scatter(x, n_before, s=7, color=C_BEFORE, alpha=0.45,
              linewidths=0, label="all points")
    a.scatter(x, n_after, s=7, color=C_AFTER, alpha=0.45,
              linewidths=0, label=f"every {RESAMPLE_STEP_M:.0f} m")
    a.set_xlabel(xlabel)
    a.set_ylabel("points per trip")
    a.legend(frameon=False, labelcolor=C_INK)

ax[1].set_title(f"vs duration   r: {table.loc['corr_with_duration'].iloc[0]:+.2f}"
                f" -> {table.loc['corr_with_duration'].iloc[1]:+.2f}",
                color=C_INK, loc="left")
ax[2].set_title(f"vs distance   r: {table.loc['corr_with_path'].iloc[0]:+.2f}"
                f" -> {table.loc['corr_with_path'].iloc[1]:+.2f}",
                color=C_INK, loc="left")
for a in ax:
    a.set_axisbelow(True)
fig.suptitle(f"{CITY} — point count before and after distance resampling",
             color=C_INK, x=0.02, ha="left")
fig.tight_layout()

In [ ]:
# the resampled frame must still satisfy the gold contract
bad = 0
for _, r in resampled.iterrows():
    c = ast.literal_eval(r.Coordinates)
    o = ast.literal_eval(r.OSMids)
    t = ast.literal_eval(r.Timestamps)
    if not (len(c) == len(o) == len(t)) or t[-1] - t[0] != r.Total_time:
        bad += 1
print("rows violating the gold contract:", bad)

if SAVE_RESAMPLED:
    out = os.path.join(DATA_DIR,
                       f"matched_trips_{CITY}_every{RESAMPLE_STEP_M:.0f}m.csv")
    resampled.to_csv(out, index=True)
    print("written", out)

## Notes

* **`Total_time` is preserved exactly.** Both endpoints survive the resampling and
  the timestamps are only interpolated in between, so the label is untouched — only
  the sequence length changes.
* **`n_points` becomes a function of distance**, which cuts both ways: a trip with a
  broken match and an absurd path length now gets *more* points, not fewer. Check
  `points_max` in the table above and look at the long tail of `path_m` before
  trusting it.
* **Arc length, not straight-line distance.** Consecutive kept points are
  `RESAMPLE_STEP_M` apart *along the road*; the straight-line gap between them is
  shorter wherever the road bends. That is the intended behaviour.
* **The share of pairs that follow the edge polyline is the quality signal** for
  the whole filter. If it is low, either the geojson is missing, the `OSMid`s do not
  match between the trips and the network, or the matching is poor — and the output
  is then mostly straight lines, which is no better than the raw fixes.
* **A pair spanning an intersection always falls back to a straight line.** With a
  cadence of one fix every 30 s in city traffic, most pairs cross at least one
  node, so do not expect the edge branch to dominate on coarse datasets.
* Cities with no coordinates at all (Quebec) are dropped by the loader with a
  warning — there is nothing to resample there.

## Self-test — no data needed

Builds a two-edge network (a 400 m arc joined to a 900 m straight), samples a trip
along it, and checks that the resampler follows the arc rather than cutting the
corner, that the marks are exactly `step` apart, and that the endpoints survive.

In [ ]:
RUN_SELF_TEST = True

if RUN_SELF_TEST:
    lat0, lon0 = 54.98, 73.37

    def _ll(dx, dy):
        return (lon0 + dx / (111320 * math.cos(math.radians(lat0))),
                lat0 + dy / 110540)

    arc = [_ll(400 * math.cos(a), 400 * math.sin(a))
           for a in np.linspace(math.pi, 0, 40)]
    strt = [_ll(400 + s, 0) for s in np.linspace(0, 900, 5)]
    t_edges = {"A": LineString(arc), "B": LineString(strt)}

    t_fwd, t_inv = make_projector(lon0, lat0)
    t_edges_m = {k: project_line(v, t_fwd) for k, v in t_edges.items()}

    rng = np.random.default_rng(0)
    raw = [arc[i] for i in (0, 13, 26, 39)] + [strt[i] for i in (1, 2, 4)]
    coords = [(p[0] + rng.normal(0, 2e-5), p[1] + rng.normal(0, 2e-5)) for p in raw]
    osmids = ["A"] * 4 + ["B"] * 3
    times = [1000, 1030, 1060, 1090, 1120, 1150, 1210]

    X, Y = t_fwd(*np.asarray(coords).T)
    xy = np.column_stack([X, Y])

    dxy, dt_, doid, k = densify_trip(xy, osmids, times, t_edges_m, MAX_PROJ_DIST_M)
    rxy, rt, roid, marks = resample_by_distance(dxy, dt_, doid, 500.0)

    print(f"pieces {k}  (3 A-A + 2 B-B along the edge, 1 straight at the corner)")
    print(f"raw polyline {seg_lengths(xy).sum():.0f} m "
          f"-> densified {seg_lengths(dxy).sum():.0f} m  (cuts no corners)")
    print(f"marks along the road {np.round(marks, 1).tolist()}")
    print(f"straight-line gaps   {np.round(seg_lengths(rxy), 1).tolist()}  "
          "(< step on the bend)")
    print(f"osmids kept {roid}")
    print(f"times {np.round(rt).astype(int).tolist()}  "
          f"(ends {times[0]} / {times[-1]} preserved)")

    assert k == {"edge": 5, "straight": 1}
    assert np.allclose(np.diff(marks)[:-1], 500.0, atol=1e-6)
    assert abs(rt[0] - times[0]) < 1e-6 and abs(rt[-1] - times[-1]) < 1e-6
    assert seg_lengths(dxy).sum() > seg_lengths(xy).sum()
    assert len(rxy) == len(rt) == len(roid)
    print("\nSELF-TEST PASSED")